# Synthetic Data Generator
### Step 9 : Row Rebuilder

In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.row_rebuilder import (
    rebuild_consumed_messages_to_observations,
)

In [ ]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "synthetic_run_001")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")

NUMBER_OF_SENSORS = int(os.getenv("SYNTHETIC_SENSOR_COUNT", "52"))
OBSERVATION_BATCH_SIZE = int(os.getenv("SYNTHETIC_OBSERVATION_BATCH_SIZE", "500"))

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = "synthetic_sensor_messages_consumed_stage"
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = "synthetic_sensor_observations_rebuilt_stage"

REBUILD_STATUS = "pending"

COMPLETE_ONLY_FLAG = True
MARK_SOURCE_REBUILT_FLAG = True

# Rebuild operates in observation rows, not sensor-message rows.
# 2500 observations = 130,000 consumed sensor messages at 52 sensors.
OBSERVATION_WINDOW_SIZE = int(os.getenv("SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE", "2500"))

print("Row rebuild config")
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("rebuild observation window:", OBSERVATION_WINDOW_SIZE)
print("expected sensors per complete observation:", NUMBER_OF_SENSORS)

---

In [3]:

engine = get_engine_from_env()


---

In [4]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(rebuild_result)

[obs-window] 1 | observation_index 1 to 2,500
[rebuild] Added 52 new columns to capstone.synthetic_sensor_observations_rebuilt_stage
[obs-window] 2 | observation_index 2,501 to 5,000
[obs-window] 3 | observation_index 5,001 to 7,500
[obs-window] 4 | observation_index 7,501 to 10,000
[obs-window] 5 | observation_index 10,001 to 12,500
[obs-window] 6 | observation_index 12,501 to 15,000
[obs-window] 7 | observation_index 15,001 to 17,500
[obs-window] 8 | observation_index 17,501 to 20,000
[obs-window] 9 | observation_index 20,001 to 22,500
[obs-window] 10 | observation_index 22,501 to 25,000
[obs-window] 11 | observation_index 25,001 to 27,500
[obs-window] 12 | observation_index 27,501 to 30,000
[obs-window] 13 | observation_index 30,001 to 32,500
[obs-window] 14 | observation_index 32,501 to 35,000
[obs-window] 15 | observation_index 35,001 to 37,500
[obs-window] 16 | observation_index 37,501 to 40,000
[obs-window] 17 | observation_index 40,001 to 42,500
[obs-window] 18 | observation_in

{'status': 'rebuilt',
 'consumed_rows': 3744000,
 'deduped_rows': 3744000,
 'rebuilt_rows': 72000,
 'rebuilt_observations': 72000,
 'updated_source_observation_groups': 72000,
 'target_table': 'synthetic_sensor_observations_rebuilt_stage'}

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM {SCHEMA}.{REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME}
    """
)

display(validation_dataframe)

,rebuilt_row_count,complete_row_count,min_observation_index,max_observation_index,min_observation_timestamp,max_observation_timestamp
0,72000,72000,1,72000,2026-04-05 08:00:00+00:00,2026-05-25 07:59:00+00:00


---

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM {SCHEMA}.{REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME}
    ORDER BY observation_index
    LIMIT 10
    """
)
display(sample_dataframe)

,dataset_id,run_id,asset_id,observation_index,observation_timestamp,stream_state,phase,meta_episode_id,meta_primary_fault_type,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,rebuild_sensor_count,rebuild_is_complete
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,1,2026-04-05 08:00:00+00:00,normal,normal,0,step_shift,2.515297,48.692947,51.770321,47.265625,582.829834,52,True
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,2,2026-04-05 08:01:00+00:00,normal,normal,0,step_shift,2.302084,44.102592,51.894260,47.265625,613.152161,52,True
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,3,2026-04-05 08:02:00+00:00,normal,normal,0,step_shift,2.492543,45.462963,52.130436,45.715290,611.710571,52,True
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,4,2026-04-05 08:03:00+00:00,normal,normal,0,step_shift,2.519502,49.937183,50.033287,43.317215,631.652649,52,True
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,5,2026-04-05 08:04:00+00:00,normal,normal,0,step_shift,2.198398,49.545708,51.425655,44.215466,619.149963,52,True
5,pump_synthetic_v1,premelt_run_001,pump_asset_001,6,2026-04-05 08:05:00+00:00,normal,normal,0,step_shift,1.650174,48.909008,50.135963,44.461655,609.747192,52,True
6,pump_synthetic_v1,premelt_run_001,pump_asset_001,7,2026-04-05 08:06:00+00:00,normal,normal,0,step_shift,NaN,50.450916,49.882420,43.635815,591.108276,52,True
7,pump_synthetic_v1,premelt_run_001,pump_asset_001,8,2026-04-05 08:07:00+00:00,normal,normal,0,step_shift,2.214998,49.661358,51.121811,44.285309,638.063416,52,True
8,pump_synthetic_v1,premelt_run_001,pump_asset_001,9,2026-04-05 08:08:00+00:00,normal,normal,0,step_shift,2.378135,50.834366,47.056705,41.732811,596.892334,52,True
9,pump_synthetic_v1,premelt_run_001,pump_asset_001,10,2026-04-05 08:09:00+00:00,normal,normal,0,step_shift,2.291894,50.491806,49.047756,44.629166,575.305298,52,True


----

In [7]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_dataframe)

,dataset_id,run_id,asset_id,observation_index,rebuild_sensor_count,rebuild_is_complete,rebuild_notes


----

In [8]:
# Dispose Engine
engine.dispose()